# ICT4403 Foundations of Generative AI — Assignment 2

Word-level autoregressive text generation on Tiny Shakespeare using Simple RNN, Bidirectional RNN, LSTM, and GRU.

Run this notebook from the **project root** (the directory that contains `config.yaml`).

## 1. Assignment objective

Predict the next word after a fixed context window, compare four recurrent architectures under matched training conditions, generate text at three temperatures, and justify the most effective model using held-out test metrics plus generated samples.

## 2. Dataset description and source

Tiny Shakespeare is a public-domain concatenation of Shakespeare plays distributed at:

https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

from src.utils import load_config, PROJECT_ROOT
from src.data_preprocessing import clean_text, tokenize, load_raw_text, Vocabulary
from src.dataset import NextTokenDataset
from src.models import MODEL_DISPLAY_NAMES

config = load_config()
dataset_meta = json.loads((PROJECT_ROOT / "data/raw/dataset_metadata.json").read_text(encoding="utf-8"))
prep_meta = json.loads((PROJECT_ROOT / "data/processed/preprocess_metadata.json").read_text(encoding="utf-8"))
dataset_meta, {k: prep_meta[k] for k in ["total_word_tokens", "vocabulary_size", "train_tokens", "val_tokens", "test_tokens"]}

## 3. Imports and configuration

In [ ]:
config["model"], config["training"], config["preprocessing"]["sequence_length"]

## 4. Text loading

In [ ]:
raw = load_raw_text(config["paths"]["raw_text"])
print("characters:", len(raw))
print(raw[:400])

## 5. Preprocessing and word-level tokenization

In [ ]:
cleaned = clean_text(raw)
tokens = tokenize(cleaned)
print(tokens[:30])
print("token count:", len(tokens))

## 6. Vocabulary creation

In [ ]:
vocab = Vocabulary.load(PROJECT_ROOT / "data/processed/vocabulary.json")
print("vocab size", len(vocab))
print("specials", vocab.index_to_token[:4])
print("UNK id", vocab.unk_index)

## 7. Input–target sequence construction

In [ ]:
ids = vocab.encode(["To", "be", "or", "not", "to", "be"])
demo = NextTokenDataset(ids, sequence_length=5)
x, y = demo[0]
print("input", [vocab.decode_token(i) for i in x.tolist()])
print("target", vocab.decode_token(int(y)))

## 8. Dataset split verification

In [ ]:
print(prep_meta["split_strategy"])
print(prep_meta["train_tokens"], prep_meta["val_tokens"], prep_meta["test_tokens"])
assert prep_meta["train_tokens"] + prep_meta["val_tokens"] + prep_meta["test_tokens"] == prep_meta["total_word_tokens"]

## 9. Architecture descriptions

- **Simple RNN:** unidirectional tanh RNN; last hidden state is classified.
- **Bidirectional RNN:** forward and backward RNNs over the *observed* window only; concatenated finals predict the next token. Not strictly causal.
- **LSTM / GRU:** gated unidirectional recurrences; last-layer hidden state is classified.

Implementations live in `src/models.py`.

## 10. Training methodology

Adam, cross-entropy, gradient clipping 1.0, five epochs unless `config.yaml` was overridden. Checkpoints are selected by validation loss. Test data is unused until after selection.

## 11. Training and validation curves

In [ ]:
history = pd.read_csv(PROJECT_ROOT / "results/metrics/training_history.csv")
display(history.head())
display(Image(str(PROJECT_ROOT / "results/figures/training_loss_curves.png")))
display(Image(str(PROJECT_ROOT / "results/figures/validation_loss_curves.png")))

## 12. Test-set evaluation

In [ ]:
test = pd.read_csv(PROJECT_ROOT / "results/metrics/test_metrics.csv")
val = pd.read_csv(PROJECT_ROOT / "results/metrics/validation_metrics.csv")
display(val)
display(test)
display(Image(str(PROJECT_ROOT / "results/figures/test_perplexity_comparison.png")))
display(Image(str(PROJECT_ROOT / "results/figures/test_accuracy_comparison.png")))

## 13. Model comparison table

In [ ]:
cols = ["model", "test_loss", "test_perplexity", "test_accuracy", "parameter_count", "total_training_time_sec"]
display(test[cols])
display(Image(str(PROJECT_ROOT / "results/figures/parameter_count_comparison.png")))
display(Image(str(PROJECT_ROOT / "results/figures/training_time_comparison.png")))

## 14. Text-generation examples

In [ ]:
samples = pd.read_csv(PROJECT_ROOT / "results/generated_text/all_samples.csv")
subset = samples[(samples["seed_phrase"] == "To be, or not to be") & (samples["temperature"] == 1.0)]
for _, row in subset.iterrows():
    display(Markdown(f"""### {row['display_name']}\n```text\n{row['full_text'][:800]}\n```"""))

## 15. Temperature analysis

In [ ]:
display(Image(str(PROJECT_ROOT / "results/figures/temperature_vs_diversity.png")))
display(Image(str(PROJECT_ROOT / "results/figures/temperature_vs_repetition.png")))
print((PROJECT_ROOT / "results/reports/temperature_analysis.md").read_text(encoding="utf-8")[:1500])

## 16. Diversity and repetition metrics

In [ ]:
pd.read_csv(PROJECT_ROOT / "results/metrics/generation_metrics.csv")

## 17. Discussion of the Bidirectional RNN

The bidirectional model encodes the complete observed window from both directions. That can improve next-token *classification* of a fully known window, but generation still has to hide the future by construction: we never feed the target. It should not be described as a causal language model. See the README section on bidirectional limitations.

## 18. Limitations

Small corpus, 20-token windows, 8k vocabulary, five-epoch budget, recurrent compression, OOV names, and possible repetitive decoding.

## 19. Conclusion

In [ ]:
best = json.loads((PROJECT_ROOT / "results/reports/best_model.json").read_text(encoding="utf-8"))
best